In [ ]:
import os
import cv2
import joblib
import numpy as np
import pandas as pd

from collections import Counter

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from imblearn.over_sampling import RandomOverSampler

from skimage.feature import graycomatrix, graycoprops

In [ ]:
TRAIN_DIR = "Dataset/train"
TEST_DIR = "Dataset/test"

CLASS_NAMES = ["organik", "non_organik"]

IMAGE_SIZE = (64, 64)

USE_BALANCING = False

K_VALUES = [1, 3, 5, 7, 9, 11, 13, 15]

DISTANCE_METRICS = [
    {"name": "euclidean", "metric": "euclidean", "p": None},
    {"name": "manhattan", "metric": "manhattan", "p": None},
    {"name": "minkowski_p3", "metric": "minkowski", "p": 3},
    {"name": "chebyshev", "metric": "chebyshev", "p": None}
]

In [ ]:
def extract_rgb_features(image_rgb):
    features = []
    for c in range(3):
        channel = image_rgb[:, :, c]
        features.append(np.mean(channel))
        features.append(np.std(channel))
    return features


def extract_glcm_features(gray_image):
    glcm = graycomatrix(
        gray_image,
        distances=[1],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=256,
        symmetric=True,
        normed=True
    )

    contrast = graycoprops(glcm, "contrast").mean()
    energy = graycoprops(glcm, "energy").mean()
    homogeneity = graycoprops(glcm, "homogeneity").mean()
    correlation = graycoprops(glcm, "correlation").mean()

    return [contrast, energy, homogeneity, correlation]


def extract_features(image_path):
    img = cv2.imread(image_path)
    img = cv2.resize(img, IMAGE_SIZE)

    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    return extract_rgb_features(rgb) + extract_glcm_features(gray)

In [ ]:
def load_dataset(base_dir):
    X, y = [], []

    for label in CLASS_NAMES:
        folder = os.path.join(base_dir, label)

        for file in os.listdir(folder):
            if file.endswith((".jpg", ".png", ".jpeg")):
                path = os.path.join(folder, file)

                try:
                    X.append(extract_features(path))
                    y.append(label)
                except:
                    pass

    return np.array(X), np.array(y)

In [5]:
X_train, y_train = load_dataset(TRAIN_DIR)
X_test, y_test = load_dataset(TEST_DIR)

print(X_train.shape, X_test.shape)
print(Counter(y_train))

(22564, 10) (2513, 10)
Counter({np.str_('organik'): 12565, np.str_('non_organik'): 9999})


In [6]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [7]:
if USE_BALANCING:
    ros = RandomOverSampler()
    X_train, y_train_enc = ros.fit_resample(X_train, y_train_enc)

In [8]:
results = []

best_model = None
best_score = 0
best_config = None

for m in DISTANCE_METRICS:
    for k in K_VALUES:

        if m["p"] is None:
            model = KNeighborsClassifier(n_neighbors=k, metric=m["metric"])
        else:
            model = KNeighborsClassifier(n_neighbors=k, metric=m["metric"], p=m["p"])

        model.fit(X_train, y_train_enc)
        pred = model.predict(X_test)

        acc = accuracy_score(y_test_enc, pred)
        f1 = f1_score(y_test_enc, pred, average="weighted")

        results.append({
            "metric": m["name"],
            "k": k,
            "acc": acc,
            "f1": f1
        })

        print(m["name"], k, acc, f1)

        if f1 > best_score:
            best_score = f1
            best_model = model
            best_config = (m["name"], k, acc, f1, pred)

euclidean 1 0.7958615200955034 0.7922614677957114
euclidean 3 0.8229208117787505 0.8200086340701837
euclidean 5 0.8209311579785118 0.8178378338313019
euclidean 7 0.8276959808993235 0.8245735331984244
euclidean 9 0.8296856346995622 0.8264518005380613
euclidean 11 0.8352566653402308 0.8320457409640271
euclidean 13 0.8368483883804219 0.8335435207396441
euclidean 15 0.8360525268603263 0.8328984734676459
manhattan 1 0.7847194588141664 0.7807653601535014
manhattan 3 0.8209311579785118 0.8176641769954234
manhattan 5 0.8241146040588938 0.8206865157152173
manhattan 7 0.8272980501392758 0.823799726831721
manhattan 9 0.828491842419419 0.8249064254154658
manhattan 11 0.8276959808993235 0.8237250408261205
manhattan 13 0.828491842419419 0.8244921489335842
manhattan 15 0.8308794269797055 0.8271649806230382
minkowski_p3 1 0.7930760047751692 0.7894517959173638
minkowski_p3 3 0.8129725427775567 0.8097417375571374
minkowski_p3 5 0.823716673298846 0.8206078562056534
minkowski_p3 7 0.8272980501392758 0.824

In [9]:
print("BEST MODEL")
print("Metric:", best_config[0])
print("K:", best_config[1])
print("Accuracy:", best_config[2])
print("F1:", best_config[3])

print("\nConfusion Matrix")
print(confusion_matrix(y_test_enc, best_config[4]))

print("\nReport")
print(classification_report(y_test_enc, best_config[4], target_names=le.classes_))

BEST MODEL
Metric: euclidean
K: 13
Accuracy: 0.8368483883804219
F1: 0.8335435207396441

Confusion Matrix
[[ 790  322]
 [  88 1313]]

Report
              precision    recall  f1-score   support

 non_organik       0.90      0.71      0.79      1112
     organik       0.80      0.94      0.86      1401

    accuracy                           0.84      2513
   macro avg       0.85      0.82      0.83      2513
weighted avg       0.85      0.84      0.83      2513



In [10]:
joblib.dump(best_model, "knn_best.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(le, "label_encoder.pkl")

['label_encoder.pkl']